# Notebook 06 — Eval set para evaluacion RAGAS

Construir un eval set de 40 preguntas con `ground_truth`, ancladas a documentos reales del corpus, para evaluar con RAGAS comparando RAG clasico vs Agentic RAG.

## Diseno del eval set

**Archivo:** `eval_set.jsonl` (un JSON por linea, easy a appendar/diff).

**Schema por entrada:**

| Campo | Tipo | Descripcion |
|---|---|---|
| `id` | str | ID unico (prefijo por categoria) |
| `categoria` | str | Bucket de la pregunta |
| `question` | str | Pregunta en español |
| `ground_truth` | str | Respuesta canonica (humano) |
| `expected_tool` | str | Tool que el agente deberia llamar |
| `fuente_esperada` | str | Doc(s) del corpus que deberian aparecer en `contexts` |

**Distribucion (40 total):**

| Categoria | N | Prefijo ID | Fuente principal |
|---|---:|---|---|
| `mundial-wiki` | 8 | MW | Wikipedia ES (selecciones, estadios, jugadores, conceptos) |
| `mundial-reglamento` | 6 | MR | Reglamento FIFA 2026 (PDF oficial) |
| `mundial-kaggle` | 6 | MK | Kaggle (Mundiales 1930-2022) |
| `mundial-calendario` | 6 | MC | Calendario 2026 (104 partidos) |
| `plataforma` | 8 | P | Corpus_91 (predicciones, rankings, tribus, referidos, monedas) |
| `multi-hop` | 4 | MH | Cruzan mundial + plataforma |
| `conversacional` | 2 | CONV | No requiere tools |

**Criterio de diseno:**
- Cada `ground_truth` viene literalmente de un doc del corpus — verificable abriendo el archivo en `fuente_esperada`.
- Multi-hop disena para forzar 2 tool_calls en paralelo (validar el fix arquitectonico de F6).
- Conversacionales validan que el agente NO llame tools cuando no hace falta.

## 1. Cargar y validar el eval set

In [1]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path

EVAL_PATH = Path.cwd() / "eval_set.jsonl" if Path.cwd().name == "notebooks" else Path.cwd() / "notebooks" / "eval_set.jsonl"
print(f"Eval set path: {EVAL_PATH}")
print(f"Existe: {EVAL_PATH.exists()}")

REQUIRED_FIELDS = {"id", "categoria", "question", "ground_truth", "expected_tool", "fuente_esperada"}
VALID_CATEGORIAS = {"mundial-wiki", "mundial-reglamento", "mundial-kaggle", "mundial-calendario", "plataforma", "multi-hop", "conversacional"}
VALID_TOOLS = {"buscar_mundial", "buscar_plataforma", "buscar_mundial+buscar_plataforma", "none"}


def load_eval_set(path):
    items = []
    with open(path, encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError as e:
                raise ValueError(f"Linea {i}: JSON invalido — {e}")
            items.append(obj)
    return items


def validate(items):
    errors = []
    seen_ids = set()
    for i, it in enumerate(items, start=1):
        missing = REQUIRED_FIELDS - set(it.keys())
        if missing:
            errors.append(f"#{i} ({it.get('id','?')}): faltan campos {missing}")
        if it.get("id") in seen_ids:
            errors.append(f"#{i}: id duplicado '{it['id']}'")
        seen_ids.add(it.get("id"))
        if it.get("categoria") not in VALID_CATEGORIAS:
            errors.append(f"#{i} ({it.get('id','?')}): categoria invalida '{it.get('categoria')}'")
        if it.get("expected_tool") not in VALID_TOOLS:
            errors.append(f"#{i} ({it.get('id','?')}): expected_tool invalido '{it.get('expected_tool')}'")
        if not it.get("question", "").strip():
            errors.append(f"#{i} ({it.get('id','?')}): question vacia")
        if not it.get("ground_truth", "").strip():
            errors.append(f"#{i} ({it.get('id','?')}): ground_truth vacio")
    return errors


items = load_eval_set(EVAL_PATH)
errors = validate(items)

print(f"\nTotal entradas: {len(items)}")
if errors:
    print(f"ERRORES: {len(errors)}")
    for e in errors:
        print(f"  {e}")
else:
    print("Schema validation: OK")

Eval set path: c:\Users\Administrador\Documents\MaestriaUSFQ\Procesamiento Lenguaje Natural\Proyecto Final\notebooks\eval_set.jsonl
Existe: True

Total entradas: 40
Schema validation: OK


## 2. Distribucion por categoria y por tool esperada

Verificamos que el balance es el que esperamos del diseno.

In [2]:
TARGET = {
    "mundial-wiki": 8,
    "mundial-reglamento": 6,
    "mundial-kaggle": 6,
    "mundial-calendario": 6,
    "plataforma": 8,
    "multi-hop": 4,
    "conversacional": 2,
}

cat_counter = Counter(it["categoria"] for it in items)
print("Distribucion por categoria:")
print(f"{'categoria':<22} {'real':>5} {'target':>7} {'match':>7}")
for cat, target in TARGET.items():
    real = cat_counter.get(cat, 0)
    ok = "OK" if real == target else "DIFF"
    print(f"{cat:<22} {real:>5} {target:>7} {ok:>7}")
print(f"{'TOTAL':<22} {sum(cat_counter.values()):>5} {sum(TARGET.values()):>7}")

print("\nDistribucion por expected_tool:")
tool_counter = Counter(it["expected_tool"] for it in items)
for tool, n in tool_counter.most_common():
    print(f"  {tool:<40} {n:>3}")

Distribucion por categoria:
categoria               real  target   match
mundial-wiki               8       8      OK
mundial-reglamento         6       6      OK
mundial-kaggle             6       6      OK
mundial-calendario         6       6      OK
plataforma                 8       8      OK
multi-hop                  4       4      OK
conversacional             2       2      OK
TOTAL                     40      40

Distribucion por expected_tool:
  buscar_mundial                            26
  buscar_plataforma                          8
  buscar_mundial+buscar_plataforma           4
  none                                       2


## 3. Muestra de cada categoria

Imprime el primer ejemplo de cada bucket para inspeccion visual.

In [3]:
def first_of(items, categoria):
    for it in items:
        if it["categoria"] == categoria:
            return it
    return None


for cat in TARGET:
    ex = first_of(items, cat)
    if not ex:
        continue
    print("=" * 70)
    print(f"[{ex['id']}] {cat}")
    print("=" * 70)
    print(f"Q: {ex['question']}")
    print(f"GT: {ex['ground_truth']}")
    print(f"Tool esperada: {ex['expected_tool']}")
    print(f"Fuente esperada: {ex['fuente_esperada']}")
    print()

[MW001] mundial-wiki
Q: ¿Cuántas selecciones participarán en la fase final del Mundial 2026?
GT: 48 selecciones participarán en la fase final del Mundial 2026.
Tool esperada: buscar_mundial
Fuente esperada: Clasificación para la Copa Mundial de Fútbol de 2026

[MR001] mundial-reglamento
Q: Según el artículo 11 del reglamento FIFA 2026, ¿cuántas selecciones participan y cómo se reparten?
GT: El artículo 11 fija en 48 las selecciones participantes: las tres anfitrionas (Canadá, México, Estados Unidos) y 45 selecciones clasificadas en la fase preliminar.
Tool esperada: buscar_mundial
Fuente esperada: ARTíCULO 11. NÚMERO DE EQUIPOS

[MK001] mundial-kaggle
Q: ¿Quién ganó el Mundial 2022 y a quién venció en la final?
GT: Argentina ganó el Mundial 2022 al vencer a Francia en la final.
Tool esperada: buscar_mundial
Fuente esperada: Copa Mundial de Fútbol de 2022

[MC001] mundial-calendario
Q: ¿En qué estadio se juega la final del Mundial 2026?
GT: La final del Mundial 2026 se juega en el MetLi

## 4. API publica del modulo

Funciones que F8 (notebook 07) va a importar. Se exponen aqui en cell para que el notebook sea autocontenido — F8 las re-defina (no hay paquete Python). El JSONL es la fuente unica de verdad.

In [4]:
def get_eval_set():
    """Devuelve la lista completa de eval items cargada desde eval_set.jsonl."""
    return load_eval_set(EVAL_PATH)


def get_by_categoria(categoria):
    """Filtra eval items por categoria."""
    return [it for it in get_eval_set() if it["categoria"] == categoria]


def get_by_id(eval_id):
    """Recupera un eval item por id."""
    for it in get_eval_set():
        if it["id"] == eval_id:
            return it
    return None


# Smoke
print(f"get_eval_set() → {len(get_eval_set())} items")
print(f"get_by_categoria('multi-hop') → {len(get_by_categoria('multi-hop'))} items")
print(f"get_by_id('MH001') → {get_by_id('MH001')['question']}")

get_eval_set() → 40 items
get_by_categoria('multi-hop') → 4 items
get_by_id('MH001') → ¿En qué estadio juega Argentina su primer partido del Mundial 2026 y cómo puedo predecir ese partido en la plataforma 91?


## 5. Resumen F7

In [5]:
print("=" * 60)
print("FASE 7 — EVAL SET LISTO")
print("=" * 60)
print(f"Archivo:         {EVAL_PATH.name}")
print(f"Total preguntas: {len(items)}")
print(f"Schema:          OK ({len(errors)} errores)")
print()
print("Por categoria:")
for cat in TARGET:
    print(f"  {cat:<22} {cat_counter.get(cat, 0)}")
print()
print("Listo para F8 (notebook 07): correr agente + RAGAS sobre eval_set.")

FASE 7 — EVAL SET LISTO
Archivo:         eval_set.jsonl
Total preguntas: 40
Schema:          OK (0 errores)

Por categoria:
  mundial-wiki           8
  mundial-reglamento     6
  mundial-kaggle         6
  mundial-calendario     6
  plataforma             8
  multi-hop              4
  conversacional         2

Listo para F8 (notebook 07): correr agente + RAGAS sobre eval_set.
